# IntelliCrash - Cloud Training Pipeline
This notebook documents the end-to-end cloud training pipeline for the IntelliCrash system. It covers data preprocessing, physics-based feature engineering, model training (XGBoost & Bi-LSTM), and exporting the optimized model for edge deployment.

## 1. Environment Setup
Mount the Google Drive filesystem to securely access the project datasets, configurations, and source code.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Dataset Extraction
Extract the compressed project archive directly to the Colab local SSD environment (`/content`). This minimizes I/O bottlenecks and significantly accelerates deep learning training times compared to reading across a network mount.

In [2]:
# Extract project to local high-speed disk
!unzip -q "/content/drive/MyDrive/IntelliCrash.zip" -d "/content/"

# Set working directory
%cd "/content/IntelliCrash"
!ls

/content/IntelliCrash
configs  data  dataset	models	outputs  requirements.txt  src


## 3. Dependency Installation
Install the required data science and machine learning libraries defined for this project.

In [8]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 18.6 MB/s eta 0:00:00


## 4. Data Preprocessing & Feature Engineering
Execute the core data pipeline to clean the raw 100Hz IMU data, apply a 2-second overlapping sliding window, calculate the physics-based Crash Severity Index (CSI), and engineer the 18 temporal/frequency domain features.

In [4]:
print("1. Preprocessing raw IMU data into 2-sec sliding windows...")
!python src/data/preprocess_imu.py

print("\n2. Engineering 18 physics features and CSI scores...")
!python src/features/feature_engineering.py

print("\n3. Generating synthetic severe/fatal crashes to balance dataset...")
!python src/data/synthetic_crashes.py

print("\n4. Exporting datasets for ML training...")
!python src/data/export_dataset_csv.py
print("\nData Pipeline Completed! All files are saved in data/processed/")

1. Preprocessing raw IMU data into 2-sec sliding windows...
Loading IMU data from: /content/IntelliCrash/dataset/df.csv
  Loaded: 1,327,597 rows, 9 columns
Cleaning IMU data...
  Removed 23,046 duplicate rows
  Label distribution: 1,304,551 non-crash, 0 crash (0.00%)
  Final: 1,304,551 rows
Creating 26,088 sliding windows (size=200, stride=50)
Windowing: 100% 26088/26088 [00:01<00:00, 25188.30win/s]
  Windows: 26,088 total, 0 crash (0.00%), 26,088 non-crash
Saved to /content/IntelliCrash/data/processed/windows/:
  imu_X.npy: (26088, 200, 6)
  imu_y.npy: (26088,)
  imu_metadata.parquet: 26088 records

Done! X shape: (26088, 200, 6), y shape: (26088,)
Crash windows: 0 / 26,088

2. Engineering 18 physics features and CSI scores...
Loading preprocessed windows...
Loaded from /content/IntelliCrash/data/processed/windows/:
  X: (26088, 200, 6), y: (26088,), metadata: 26088 records

Extracting 18 features from 26,088 windows...
Extracting features: 100% 26088/26088 [00:13<00:00, 1901.88win/s]

## 5. Phase-1 Training: Rash Driving Classifier
Train an XGBoost classifier on the extracted feature set to accurately identify and filter out rash driving maneuvers (e.g., hard braking, swerving) from actual collision events.

In [5]:
!python src/models/train_rash_classifier.py

Loading engineered features...
Loading metadata...
Data loaded: X=(26088, 18), y=(26088,)

Class distribution:
  Normal: 26058 (99.89%)
  Lane Weaving: 13 (0.05%)
  Hard Cornering: 17 (0.07%)

Training set: 20870 samples
Testing set: 5218 samples

Training XGBoost Classifier...
[0]	validation_0-mlogloss:0.00906	validation_1-mlogloss:0.00933
[10]	validation_0-mlogloss:0.00412	validation_1-mlogloss:0.00680
[20]	validation_0-mlogloss:0.00210	validation_1-mlogloss:0.00629
[30]	validation_0-mlogloss:0.00127	validation_1-mlogloss:0.00638
[40]	validation_0-mlogloss:0.00085	validation_1-mlogloss:0.00646
[50]	validation_0-mlogloss:0.00062	validation_1-mlogloss:0.00670
[60]	validation_0-mlogloss:0.00049	validation_1-mlogloss:0.00696
[70]	validation_0-mlogloss:0.00040	validation_1-mlogloss:0.00707
[80]	validation_0-mlogloss:0.00033	validation_1-mlogloss:0.00716
[90]	validation_0-mlogloss:0.00029	validation_1-mlogloss:0.00724
[99]	validation_0-mlogloss:0.00026	validation_1-mlogloss:0.00731

Evalua

## 6. Phase-2 Training: Bi-LSTM Crash Detection
Train the primary Bidirectional Long Short-Term Memory (Bi-LSTM) neural network using the augmented dataset to classify crash events and predict severity based on sequential spatial-temporal IMU patterns.

In [13]:
!python src/models/train_lstm.py --epochs 10

Using device: cuda
Loading augmented IMU dataset...
Extracting features for 37,268 windows (this may take a minute)...
Extracting features: 100% 37268/37268 [00:20<00:00, 1824.55win/s]
Final input shape: (37268, 200, 24)
Splits - Train: 26086, Val: 5591, Test: 5591
Epoch 01: Train Loss=0.0183 Acc=0.9563 | Val Loss=0.0106 Acc=0.9666
  -> Saved new best model
Epoch 02: Train Loss=0.0106 Acc=0.9748 | Val Loss=0.0072 Acc=0.9827
  -> Saved new best model
Epoch 03: Train Loss=0.0096 Acc=0.9786 | Val Loss=0.0065 Acc=0.9819
  -> Saved new best model
Epoch 04: Train Loss=0.0085 Acc=0.9804 | Val Loss=0.0129 Acc=0.9637
Epoch 05: Train Loss=0.0080 Acc=0.9812 | Val Loss=0.0055 Acc=0.9857
  -> Saved new best model
Epoch 06: Train Loss=0.0070 Acc=0.9838 | Val Loss=0.0050 Acc=0.9862
  -> Saved new best model
Epoch 07: Train Loss=0.0066 Acc=0.9838 | Val Loss=0.0044 Acc=0.9893
  -> Saved new best model
Epoch 08: Train Loss=0.0061 Acc=0.9860 | Val Loss=0.0042 Acc=0.9893
  -> Saved new best model
Epoch 09

## 7. Model Evaluation & Edge Deployment Export
Evaluate the dual-stage fusion system (0.6 ML confidence + 0.4 physics CSI score) on the test split. Finally, export the trained PyTorch model to ONNX format to enable efficient, low-latency inference on the target Raspberry Pi edge device.

In [14]:
print("Exporting PyTorch model to ONNX format...")
!python src/models/export_onnx.py

Exporting PyTorch model to ONNX format...
Loaded weights from /content/IntelliCrash/models/checkpoints/best_lstm.pth
/content/IntelliCrash/src/models/export_onnx.py:39: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0606 13:43:45.915000 18193 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
[torch.onnx] Obtain model graph for `IntelliCrashLSTM([...]` with `torch.export.export(..., 

In [17]:
print("\nEvaluating the 0.6 ML + 0.4 CSI Fusion Gate System...")
!python src/models/evaluate_ablation.py


Evaluating the 0.6 ML + 0.4 CSI Fusion Gate System...
Loading test data...
Loading trained Bi-LSTM model...
Running Inference on Test Set...

-----------------------------------------------------------------
Architecture Configuration     | Recall       | FPR         
-----------------------------------------------------------------
1. Physics CSI Gate Only       |   30.2%      |    0.0%
2. Bi-LSTM Only                |   98.0%      |    0.4%
3. Hybrid Fusion Gate          |   96.7%      |    0.1%
-----------------------------------------------------------------


In [20]:
import numpy as np, torch
from pathlib import Path
from src.models.lstm import IntelliCrashLSTM
from src.features.feature_engineering import compute_csi
from sklearn.metrics import confusion_matrix

data_dir = Path("data/processed/windows")
X_test = np.load(data_dir / "imu_augmented_X_test.npy")
y_test = np.load(data_dir / "imu_augmented_y_test.npy")
features_test = X_test[:, 0, 6:]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = IntelliCrashLSTM(input_size=24, hidden_size=128, num_layers=2).to(device)
model.load_state_dict(torch.load("models/checkpoints/best_lstm.pth", map_location=device))
model.eval()

lstm_probs = []
with torch.no_grad():
    for i in range(0, len(X_test), 256):
        batch = torch.FloatTensor(X_test[i:i+256]).to(device)
        probs, _ = model(batch)
        lstm_probs.append(probs.cpu().numpy().flatten())
lstm_probs = np.concatenate(lstm_probs)
csi_scores = compute_csi(features_test, mode="real_car")

print("-" * 70)
print(f"{'Fusion Weights (LSTM:CSI)':<28} | {'Recall':<12} | {'FPR':<12}")
print("-" * 70)

for lstm_w, csi_w in [(1.0, 0.0), (0.7, 0.3), (0.6, 0.4), (0.5, 0.5), (0.4, 0.6), (0.0, 1.0)]:
    fusion = lstm_w * lstm_probs + csi_w * csi_scores
    preds = (fusion > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    recall = 100 * tp / (tp + fn)
    fpr = 100 * fp / (fp + tn)
    label = f"{lstm_w:.1f} : {csi_w:.1f}"
    print(f"{label:<28} | {recall:>6.1f}%      | {fpr:>6.1f}%")

print("-" * 70)


----------------------------------------------------------------------
Fusion Weights (LSTM:CSI)    | Recall       | FPR         
----------------------------------------------------------------------
1.0 : 0.0                    |   98.0%      |    0.4%
0.7 : 0.3                    |   97.1%      |    0.2%
0.6 : 0.4                    |   96.7%      |    0.1%
0.5 : 0.5                    |   95.5%      |    0.1%
0.4 : 0.6                    |   78.8%      |    0.1%
0.0 : 1.0                    |   30.2%      |    0.0%
----------------------------------------------------------------------


## 8. Checkpoint Persistence
Copy the trained model checkpoints, compiled ONNX files, and the generated preprocessed CSV dataset from the ephemeral local disk back to the permanent Google Drive storage layer.

In [24]:
import shutil, os

dst = "/content/drive/MyDrive/IntelliCrash_Trained_Models"

# Delete old folder if exists, start fresh
if os.path.exists(dst):
    shutil.rmtree(dst)

# Copy everything in one shot
shutil.copytree("models", f"{dst}/models")
shutil.copytree("data/processed", f"{dst}/data_processed")
shutil.copy("IntelliCrash_Dataset.csv", f"{dst}/IntelliCrash_Dataset.csv")

print("DONE! Everything saved:")
for root, dirs, files in os.walk(dst):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / (1024*1024)
        print(f"  {path.replace(dst, '')} — {size:.1f} MB")


DONE! Everything saved:
  /IntelliCrash_Dataset.csv — 7.1 MB
  /models/onnx/intellicrash_lstm.onnx.data — 2.3 MB
  /models/onnx/intellicrash_lstm.onnx — 0.1 MB
  /models/checkpoints/xgboost_rash_classifier.json — 0.3 MB
  /models/checkpoints/best_lstm.pth — 2.3 MB
  /models/checkpoints/rash_label_encoder.pkl — 0.0 MB
  /data_processed/imu_features.npy — 1.8 MB
  /data_processed/imu_csi_scores.npy — 0.1 MB
  /data_processed/windows/imu_augmented_metadata.parquet — 0.0 MB
  /data_processed/windows/imu_augmented_y.npy — 0.1 MB
  /data_processed/windows/imu_X.npy — 119.4 MB
  /data_processed/windows/imu_metadata.parquet — 0.9 MB
  /data_processed/windows/imu_y.npy — 0.1 MB
  /data_processed/windows/imu_augmented_y_test.npy — 0.0 MB
  /data_processed/windows/imu_augmented_X_test.npy — 102.4 MB
  /data_processed/windows/imu_augmented_X.npy — 170.6 MB
